# Universal Precision Runtime (UPR) — Notebook 01
## BitPlane Model Conversion & Level 1 Weight Reconstruction Verification

---

> **Prerequisite:** Run `00_bootstrap_upr_files.ipynb` once before this notebook.

### Objective
1. Download original FP16 baseline model (`Qwen/Qwen3.5-0.8B`).
2. Convert every parameter tensor into packed bit-plane files (`plane15.bin` through `plane0.bin`) with packing size assertions (Fix 8).
3. Reconstruct full 16-bit FP16 weights from bit-planes.
4. Validate Stage 1 Success Criterion: **100% exact bitwise tensor equality (`torch.equal == True`)** across all parameters and log per-tensor stats to `results/reconstruction.csv` (Fix 5).

### Step 1: Environment Setup & Import

In [ ]:
import os
import sys
import gc
import time
import importlib
import torch

# Set Hugging Face Token safely from environment or Colab secrets
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    if not HF_TOKEN:
        HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
    DRIVE_DIR = '/content/drive/MyDrive/UniversalPrecisionRuntime'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    os.chdir(DRIVE_DIR)
except ImportError:
    print('Running in local environment.')

WORK_DIR = os.getcwd()
print(f'Active Working Directory: {WORK_DIR}')
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

!pip install -q transformers accelerate huggingface_hub torch numpy tqdm datasets psutil

import upr
importlib.reload(upr)
upr.set_seed(42)

from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)

print(f'UPR {upr.__version__} ready. PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

### Step 2: Convert Model to BitPlane Format
Convert `Qwen/Qwen3.5-0.8B` into 16 packed bit planes per tensor.

In [ ]:
MODEL_ID = 'Qwen/Qwen3.5-0.8B'
OUTPUT_DIR = 'models/bitplane_qwen'

timer = upr.IsolatedTimer()
timer.start("bitplane_conversion")

upr.convert_to_bitplanes(
    model_or_path=MODEL_ID,
    output_directory=OUTPUT_DIR,
    torch_dtype=torch.float16
)

conv_time = timer.stop("bitplane_conversion")
print(f'BitPlane conversion completed in {conv_time:.2f} seconds.')

### Step 3: Level 1 Validation — Exact 16-Bit Reconstruction
Verify exact bitwise match (`torch.equal`) for all tensors and log per-tensor stats to `results/reconstruction.csv`.

In [ ]:
from transformers import AutoModelForCausalLM

print('Loading original FP16 baseline model on cpu for comparison...')
orig_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, low_cpu_mem_usage=True)
orig_state_dict = orig_model.state_dict()

print('Reconstructing 16-bit state dict...')
timer.start("tensor_reconstruction")
recon_state_dict = upr.BitPlaneModel.load_reconstructed_state_dict(
    bitplane_directory=OUTPUT_DIR,
    bits=16,
    device='cpu',
    export_reconstruction_csv=True,
    original_state_dict=orig_state_dict,
    csv_output_path="results/reconstruction.csv"
)
recon_time = timer.stop("tensor_reconstruction")

total_tensors = len(orig_state_dict)
exact_matches = 0
for name, orig_t in orig_state_dict.items():
    recon_t = recon_state_dict[name]
    if torch.equal(orig_t, recon_t):
        exact_matches += 1

print('='*60)
print('LEVEL 1 RECONSTRUCTION RESULTS (16-bit Full Reconstruction)')
print(f'Total Parameter Tensors Verified: {total_tensors}')
print(f'Exact Bitwise Matches (torch.equal == True): {exact_matches} / {total_tensors} ({exact_matches/total_tensors*100:.2f}%)')
print(f'Reconstruction Time: {recon_time:.2f} seconds')
print('Per-tensor metrics exported to: results/reconstruction.csv')
print('='*60)